# Gradient Flow ODE Simulator — Full Replication

**Goals per run:**
- **Goal 1** — Count bias clusters and inflection points of the target
- **Goal 3** — Verify stationarity: plug final $(a,b)$ back into the ODEs, confirm velocities $\approx 0$, confirm the integrated residual $R_j = \int_{b_j}^{1}(f-f^*)\,dx \approx 0$ for every active neuron, and check that the number of active neurons never exceeds the inflection point count $k$

**Model:** $f(x) = \sum_{j=1}^m a_j \sigma(x - b_j)$,  $\sigma(z)=\max(0,z)$,  $x\in[-1,1]$

**ODEs:**
$$\dot{a}_j = -\int_{-1}^{1}(f-f^*)\,\sigma(x-b_j)\,dx \qquad \dot{b}_j = a_j\int_{b_j}^{1}(f-f^*)\,dx$$

Figures saved to: `figures/Replication data/{target}/m={m}/T={T}/`  
A run is **skipped** only when all four output files already exist.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import os, csv

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

## Core Functions

In [ ]:
N_QUAD = 400
X_QUAD = np.linspace(-1, 1, N_QUAD)
DX     = X_QUAD[1] - X_QUAD[0]

def relu(z):
    return np.maximum(0.0, z)

def network(x, a, b):
    return (a * relu(x[:, None] - b[None, :])).sum(axis=1)

def make_ode(m, f_star):
    fstar_vals = f_star(X_QUAD)
    def ode(t, y):
        a, b      = y[:m], y[m:]
        residual  = network(X_QUAD, a, b) - fstar_vals
        relu_mat  = relu(X_QUAD[:, None] - b[None, :])
        da        = -(residual[:, None] * relu_mat).sum(0) * DX
        cum_right = np.cumsum(residual[::-1])[::-1] * DX
        idx       = np.searchsorted(X_QUAD, b).clip(0, N_QUAD - 1)
        db        = a * cum_right[idx]
        return np.concatenate([da, db])
    return ode

def compute_ode_velocities(a, b, f_star):
    """Return (da, db, R). R[j] = integral of residual from b[j] to 1."""
    fstar_vals = f_star(X_QUAD)
    residual   = network(X_QUAD, a, b) - fstar_vals
    relu_mat   = relu(X_QUAD[:, None] - b[None, :])
    da         = -(residual[:, None] * relu_mat).sum(0) * DX
    cum_right  = np.cumsum(residual[::-1])[::-1] * DX
    idx        = np.searchsorted(X_QUAD, b).clip(0, N_QUAD - 1)
    R          = cum_right[idx]
    db         = a * R
    return da, db, R

def count_clusters(biases, tol=0.02):
    sorted_b = np.sort(biases)
    return 1 + int(np.sum(np.diff(sorted_b) > tol))

def get_cluster_centers(biases, tol=0.02):
    sorted_b = np.sort(biases)
    centers, group = [], [sorted_b[0]]
    for i in range(1, len(sorted_b)):
        if sorted_b[i] - sorted_b[i-1] > tol:
            centers.append(np.mean(group))
            group = []
        group.append(sorted_b[i])
    centers.append(np.mean(group))
    return np.array(centers)

def count_inflection_points(f_star, n=2000):
    x   = np.linspace(-1, 1, n)
    d2y = np.gradient(np.gradient(f_star(x), x), x)
    return int(np.sum(np.diff(np.sign(d2y)) != 0))

## Target Functions & Sweep Parameters

In [ ]:
TARGETS = {
    'x2':              (r'$x^2$',                            lambda x: x**2),
    'sin2pi_05sin4pi': (r'$\sin(2\pi x)+0.5\sin(4\pi x)$',  lambda x: np.sin(2*np.pi*x) + 0.5*np.sin(4*np.pi*x)),
    'x3_minus_x':      (r'$x^3-x$',                         lambda x: x**3 - x),
    'exp_x':           (r'$e^x$',                            lambda x: np.exp(x)),
    'sin6pi':          (r'$\sin(6\pi x)$',                   lambda x: np.sin(6*np.pi*x)),
}

M_VALUES = [100, 500, 1000, 1500]
T_VALUES = [80, 200, 400, 800, 1000]
N_SAVE   = 500
SEED     = 42
ACTIVE_THRESHOLD = 0.05  # neuron active if |a_j| > threshold * max|a_j|

FIG_BASE = os.path.join('..', 'figures', 'Replication data')

total = len(TARGETS) * len(M_VALUES) * len(T_VALUES)
print(f'Total runs planned : {total}')
print(f'Targets            : {list(TARGETS.keys())}')
print(f'm values           : {M_VALUES}')
print(f'T values           : {T_VALUES}')
print(f'Active threshold   : |a_j| > {ACTIVE_THRESHOLD} * max|a_j|')

## Main Simulation Loop

**Output files per run** (all saved to `Replication data/{target}/m={m}/T={T}/`):

| File | Goal | Contents |
|---|---|---|
| `slide93_reproduction.png` | 1 | Bias trajectories, final fit, loss curve |
| `clusters_vs_inflections.png` | 1 | Cluster locations overlaid on target |
| `ode_verification.png` | 3 | $|\dot{a}_j|$, $|\dot{b}_j|$, and $R_j$ at convergence |
| `convergence_check.csv` | 3 | Per-neuron $\dot{a}_j$, $\dot{b}_j$, $R_j$, active flag |

A run is **skipped** only if all four files exist.

In [ ]:
done = 0
skipped = 0

for target_key, (target_label, f_star) in TARGETS.items():
    n_inf = count_inflection_points(f_star)

    for m in M_VALUES:
        for T in T_VALUES:
            done += 1

            out_dir = os.path.join(FIG_BASE, target_key, f'm={m}', f'T={T}')
            f_traj  = os.path.join(out_dir, 'slide93_reproduction.png')
            f_clust = os.path.join(out_dir, 'clusters_vs_inflections.png')
            f_ode   = os.path.join(out_dir, 'ode_verification.png')
            f_csv   = os.path.join(out_dir, 'convergence_check.csv')

            if all(os.path.exists(p) for p in [f_traj, f_clust, f_ode, f_csv]):
                skipped += 1
                print(f'[{done}/{total}] SKIP  {target_key}  m={m}  T={T}')
                continue

            print(f'[{done}/{total}] RUN   {target_key}  m={m}  T={T} ...', end=' ', flush=True)
            os.makedirs(out_dir, exist_ok=True)

            np.random.seed(SEED)
            a0  = np.random.randn(m) * 0.01
            b0  = np.random.uniform(-1, 1, m)
            sol = solve_ivp(
                make_ode(m, f_star),
                t_span=(0, T),
                y0=np.concatenate([a0, b0]),
                method='RK45',
                t_eval=np.linspace(0, T, N_SAVE),
                rtol=1e-4, atol=1e-6,
                max_step=max(0.1, T / 500),
            )

            fstar_vals = f_star(X_QUAD)
            losses = np.array([
                0.5 * np.trapezoid(
                    (network(X_QUAD, sol.y[:m, i], sol.y[m:, i]) - fstar_vals)**2,
                    X_QUAD)
                for i in range(sol.y.shape[1])
            ])

            a_final         = sol.y[:m, -1]
            b_final         = sol.y[m:, -1]
            x_plot          = np.linspace(-1, 1, 500)
            f_final         = network(x_plot, a_final, b_final)
            n_clusters      = count_clusters(b_final)
            cluster_centers = get_cluster_centers(b_final)

            # Goal 3: stationarity quantities
            da_final, db_final, R_final = compute_ode_velocities(a_final, b_final, f_star)
            a_max     = np.abs(a_final).max() if np.abs(a_final).max() > 0 else 1.0
            is_active = np.abs(a_final) > ACTIVE_THRESHOLD * a_max
            n_active  = int(is_active.sum())

            sort_idx = np.argsort(b_final)
            b_s      = b_final[sort_idx]
            a_s      = a_final[sort_idx]
            da_s     = da_final[sort_idx]
            db_s     = db_final[sort_idx]
            R_s      = R_final[sort_idx]
            active_s = is_active[sort_idx]

            # Figure 1: bias trajectories / final fit / loss  (Goal 1)
            if not os.path.exists(f_traj):
                fig, axes = plt.subplots(1, 3, figsize=(15, 4))
                fig.suptitle(f'target={target_label},  m={m},  T={T}', fontsize=12)
                ax = axes[0]
                for j in range(m):
                    ax.plot(sol.t, np.sort(sol.y[m:, :], axis=0)[j],
                            color='steelblue', alpha=min(0.3, 15/m), linewidth=0.5)
                ax.set_xlabel('Time'); ax.set_ylabel('Bias value')
                ax.set_title('Sorted Bias Trajectories'); ax.set_xlim([0, T])
                ax = axes[1]
                ax.plot(x_plot, f_star(x_plot), 'k--', lw=2, label=f'Target {target_label}')
                ax.plot(x_plot, f_final, 'r-', lw=2, label='Network $f$')
                ax.scatter(b_final, np.zeros_like(b_final), c='steelblue', s=8, zorder=5)
                ax.set_xlabel('$x$'); ax.set_title('Final Fit vs Target')
                ax.legend(fontsize=8); ax.set_xlim([-1, 1])
                ax = axes[2]
                ax.semilogy(sol.t, losses, color='darkorange', lw=1.5)
                ax.set_xlabel('Time'); ax.set_ylabel('MSE Loss')
                ax.set_title(f'Loss (log)   final={losses[-1]:.2e}')
                ax.set_xlim([0, T]); ax.grid(True, alpha=0.3)
                plt.tight_layout()
                plt.savefig(f_traj, bbox_inches='tight')
                plt.close()

            # Figure 2: cluster locations vs inflection points  (Goal 1)
            if not os.path.exists(f_clust):
                x_fine = np.linspace(-1, 1, 2000)
                d2f    = np.gradient(np.gradient(f_star(x_fine), x_fine), x_fine)
                infl_x = x_fine[:-1][np.diff(np.sign(d2f)) != 0]
                fig, ax = plt.subplots(figsize=(10, 4))
                fig.suptitle(
                    f'target={target_label},  m={m},  T={T}  '
                    f'|  clusters={n_clusters},  inflections={n_inf}', fontsize=11)
                ax.plot(x_fine, f_star(x_fine), 'k-', lw=2, label=f'Target {target_label}')
                ax.plot(x_plot, f_final, 'r-', lw=1.5, label='Final network')
                for cx in cluster_centers:
                    ax.axvline(cx, color='steelblue', alpha=0.6, lw=0.8)
                for ix in infl_x:
                    ax.axvline(ix, color='green', alpha=0.5, lw=1.0, linestyle='--')
                from matplotlib.lines import Line2D
                h, _ = ax.get_legend_handles_labels()
                h += [Line2D([0],[0], color='steelblue', lw=1.5, label=f'Bias clusters ({n_clusters})'),
                      Line2D([0],[0], color='green', lw=1.5, ls='--', label=f'Inflection pts ({n_inf})')]
                ax.legend(handles=h, fontsize=9)
                ax.set_xlabel('$x$'); ax.set_title('Cluster Locations vs Inflection Points')
                plt.tight_layout()
                plt.savefig(f_clust, bbox_inches='tight')
                plt.close()

            # Figure 3: ODE stationarity check  (Goal 3)
            if not os.path.exists(f_ode):
                fig, axes = plt.subplots(1, 3, figsize=(15, 4))
                fig.suptitle(
                    f'Stationarity check — target={target_label},  m={m},  T={T}\n'
                    f'active neurons={n_active},  inflections={n_inf},  '
                    f'active <= inflections: {n_active <= n_inf}',
                    fontsize=10)
                ax = axes[0]
                ax.scatter(b_s, np.abs(da_s) + 1e-15, c=np.abs(a_s), cmap='viridis', s=12)
                ax.set_xlabel('Bias $b_j$'); ax.set_ylabel('$|\\dot{a}_j|$')
                ax.set_title('Amplitude velocity at $t=T$\n(should be $\\approx 0$)')
                ax.set_yscale('log'); ax.grid(True, alpha=0.3)
                ax = axes[1]
                ax.scatter(b_s, np.abs(db_s) + 1e-15, c=np.abs(a_s), cmap='viridis', s=12)
                ax.set_xlabel('Bias $b_j$'); ax.set_ylabel('$|\\dot{b}_j|$')
                ax.set_title('Bias velocity at $t=T$\n(should be $\\approx 0$)')
                ax.set_yscale('log'); ax.grid(True, alpha=0.3)
                ax = axes[2]
                sizes = np.clip(np.abs(a_s) / (a_max + 1e-12) * 80, 4, 80)
                sc = ax.scatter(b_s, R_s, c=np.abs(a_s), cmap='viridis', s=sizes)
                ax.scatter(b_s[active_s], R_s[active_s],
                           edgecolors='red', facecolors='none', s=70, lw=1.3,
                           label=f'Active ({n_active}), $R_j\\approx 0$ expected')
                ax.axhline(0, color='k', lw=1.0, linestyle='--')
                plt.colorbar(sc, ax=ax, label='$|a_j|$')
                ax.set_xlabel('Bias $b_j$')
                ax.set_ylabel('$R_j = \\int_{b_j}^{1}(f-f^*)\\,dx$')
                ax.set_title(f'Integrated residual\nactive={n_active} <= k={n_inf}: {n_active<=n_inf}')
                ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
                plt.tight_layout()
                plt.savefig(f_ode, bbox_inches='tight')
                plt.close()

            # CSV: per-neuron stationarity data  (Goal 3)
            if not os.path.exists(f_csv):
                R_tol = 1e-3
                with open(f_csv, 'w', newline='') as csvfile:
                    writer = csv.writer(csvfile)
                    writer.writerow(['neuron_idx','b_j','a_j','da_dt','db_dt','R_j','is_active','R_near_zero'])
                    for i in range(m):
                        writer.writerow([
                            sort_idx[i],
                            f'{b_s[i]:.6f}', f'{a_s[i]:.6f}',
                            f'{da_s[i]:.6e}', f'{db_s[i]:.6e}', f'{R_s[i]:.6e}',
                            int(active_s[i]), int(abs(R_s[i]) < R_tol),
                        ])

            print(f'clusters={n_clusters}, active={n_active}, inflections={n_inf}, '
                  f'active<=k: {n_active<=n_inf},  '
                  f'max|da|={np.abs(da_final).max():.2e},  '
                  f'max|db|={np.abs(db_final).max():.2e},  '
                  f'loss={losses[-1]:.5f}')

print(f'\nFinished. {done-skipped} runs completed, {skipped} skipped.')
